# Day 04：反向传播的直觉 —— 链式法则的代码实现

> ☀️ 第二周 · 破局与复兴 · 第 4 天

昨天我们学了损失函数和梯度下降——模型知道了「错得多离谱」和「怎么改」。

但还有一个核心问题没解决：**输出层的误差，怎么知道该归咎于哪个隐藏神经元？**

这叫「信用分配问题」——输出误差该**分摊**给谁？

**反向传播（Backpropagation）** 就是解决这个问题的算法。它的核心是**链式法则**。

**今天的任务**：用代码直观理解反向传播是如何工作的。

---
## 1. 历史背景：15 年的等待

1969 年 Minsky 证明了单层感知机无法解决 XOR 后，AI 进入寒冬。人们知道「加一层隐藏层」可能解决问题，但不知道**怎么训练**这个多层网络。

核心困难是：**输出层的误差怎么传回隐藏层？**

1974 年，Paul Werbos 在博士论文中首次提出了用链式法则训练多层网络的方法，但没有引起关注。

直到 1986 年，Rumelhart、Hinton 和 Williams 发表了著名的论文《Learning representations by back-propagating errors》，反向传播算法才真正被学术界接受。

这篇论文发表在 Nature 上，直接引发了神经网络的复兴。Hinton 后来被称为「深度学习之父」。

---
## 2. 生活隐喻：公司追责

### 信用分配问题

想象一家公司的产品出了质量问题，客户投诉了。

```
原材料 → 工人A加工 → 工人B组装 → 质检 → 产品 → 客户投诉
```

老板需要追责：这个质量问题是谁造成的？
- 是原材料有瑕疵？
- 是工人A加工不当？
- 是工人B组装失误？
- 还是质检没检出来？

反向传播就是一套**逐层追责**的算法：从客户投诉（输出误差）开始，一层一层往回追溯，计算每个环节应承担的责任比例。

### 链式法则 = 多米诺骨牌

假设 A 倒了会推倒 B，B 倒了会推倒 C。

现在 C 倒了（输出误差），我们要追究 A 的责任：
- A 对 C 的影响 = A 对 B 的影响 × B 对 C 的影响

这就是**链式法则**——把一长串因果关系拆成小段，逐段计算，然后相乘。

---
## 3. 数学直觉：链式法则

### 基本形式

如果 $C$ 依赖于 $B$，$B$ 依赖于 $A$，那么：

$$\frac{\partial C}{\partial A} = \frac{\partial C}{\partial B} \times \frac{\partial B}{\partial A}$$

用大白话说：**A 对 C 的影响 = A 对 B 的影响 × B 对 C 的影响**。

### 在神经网络中

一个简单的计算链：

$$x \xrightarrow{\times 2} z \xrightarrow{+1} y \xrightarrow{(y-3)^2} C$$

用代码验证链式法则：

In [ ]:
import torch

# 定义计算链：x → z → y → C
x = torch.tensor(1.0, requires_grad=True)  # 起始值 x=1
z = 2 * x        # z = 2 * x = 2
y = z + 1        # y = z + 1 = 3
C = (y - 3) ** 2  # C = (y-3)² = 0

print("前向传播：")
print(f"  x = {x.item()}")
print(f"  z = 2 * x = {z.item()}")
print(f"  y = z + 1 = {y.item()}")
print(f"  C = (y-3)² = {C.item()}")

# 反向传播：PyTorch 自动计算梯度
C.backward()

print("\n反向传播（链式法则）：")
print(f"  ∂C/∂y = 2*(y-3) = {2*(y.item()-3)}")
print(f"  ∂y/∂z = 1")
print(f"  ∂z/∂x = 2")
print(f"  ∂C/∂x = ∂C/∂y × ∂y/∂z × ∂z/∂x = {2*(y.item()-3)} × 1 × 2")
print(f"  PyTorch 计算的 ∂C/∂x = {x.grad.item()}")
print(f"  手动计算的 ∂C/∂x = {2*(y.item()-3) * 1 * 2}")

---
## 4. 反向传播的分步实现

现在把链式法则应用到一个神经网络中。

### 一层网络的前向传播

```
输入 x ──→ [z = x·W + b] ──→ [a = sigmoid(z)] ──→ 输出 a
```

前向传播就是从左到右算：$x \to z \to a$

反向传播就是从右到左算梯度：$\frac{\partial L}{\partial a} \to \frac{\partial L}{\partial z} \to \frac{\partial L}{\partial W}$

关键公式（Sigmoid 激活）：

$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \times \sigma'(z) = \frac{\partial L}{\partial a} \times a(1-a)$$

$$\frac{\partial L}{\partial W} = x^T \times \frac{\partial L}{\partial z}$$

$$\frac{\partial L}{\partial b} = \sum \frac{\partial L}{\partial z}$$

用代码实现一个支持前向和反向传播的神经元：

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

def sigmoid_grad(a):
    """Sigmoid 的导数：σ'(z) = σ(z) * (1 - σ(z)) = a * (1-a)
    参数: a — sigmoid(z) 的输出值
    返回: sigmoid 在 z 处的导数值
    """
    return a * (1 - a)

class SimpleNeuron:
    """单个神经元：支持前向传播和手动反向传播

    结构：输入 x → 线性变换 z=x·W+b → 激活 a=sigmoid(z) → 输出
    """

    def __init__(self, input_size):
        """初始化权重和偏置（随机）"""
        self.W = torch.randn(input_size, 1)  # 权重 [input_size, 1]
        self.b = torch.randn(1, 1)  # 偏置 [1, 1]

        # 保存中间值（反向传播时需要）
        self.x = None  # 输入
        self.z = None  # 线性变换输出
        self.a = None  # 激活输出

    def forward(self, x):
        """前向传播：x → z → a
        参数: x — 输入 [batch_size, input_size]
        返回: a — 激活后输出 [batch_size, 1]
        """
        self.x = x.clone()  # 保存输入（反向传播时需要）
        self.z = torch.matmul(x, self.W) + self.b  # 线性变换
        self.a = sigmoid(self.z)  # Sigmoid 激活
        return self.a

    def backward(self, grad_output):
        """反向传播：计算 ∂L/∂W 和 ∂L/∂b
        参数: grad_output — 从上一层传来的梯度 ∂L/∂a [batch_size, 1]
        返回: (grad_W, grad_b) — 权重和偏置的梯度
        """
        # 第 1 步：∂L/∂z = ∂L/∂a × σ'(z) = grad_output × a*(1-a)
        grad_z = grad_output * sigmoid_grad(self.a)  # [batch, 1]

        # 第 2 步：∂L/∂W = x^T × ∂L/∂z
        grad_W = torch.matmul(self.x.T, grad_z)  # [input_size, 1]

        # 第 3 步：∂L/∂b = sum(∂L/∂z)
        grad_b = grad_z.sum(dim=0, keepdim=True)  # [1, 1]

        return grad_W, grad_b

print("SimpleNeuron 创建完成")
print("  forward(x)  — 前向传播")
print("  backward(grad) — 反向传播，返回 (grad_W, grad_b)")

### 用这个神经元训练 AND 逻辑

AND 是线性可分的，单个神经元就能学会。用它来验证反向传播是否正确。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

def sigmoid_grad(a):
    """Sigmoid 导数"""
    return a * (1 - a)

class SimpleNeuron:
    """单个神经元"""
    def __init__(self, input_size):
        self.W = torch.randn(input_size, 1)
        self.b = torch.randn(1, 1)
        self.x = None
        self.z = None
        self.a = None

    def forward(self, x):
        """前向传播"""
        self.x = x.clone()
        self.z = torch.matmul(x, self.W) + self.b
        self.a = sigmoid(self.z)
        return self.a

    def backward(self, grad_output):
        """反向传播"""
        grad_z = grad_output * sigmoid_grad(self.a)
        grad_W = torch.matmul(self.x.T, grad_z)
        grad_b = grad_z.sum(dim=0, keepdim=True)
        return grad_W, grad_b

# AND 数据：只有两个都是 1 时才输出 1
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [0.0], [0.0], [1.0]])

# 创建神经元
neuron = SimpleNeuron(input_size=2)
lr = 5.0  # 学习率

print("训练 AND 逻辑（单个神经元 + 手动反向传播）")
print("=" * 55)

for epoch in range(100):
    # 前向传播
    y_pred = neuron.forward(X)  # [4, 1]

    # 计算损失（MSE）
    loss = torch.mean((y_pred - y) ** 2)

    # 反向传播
    grad_output = 2 * (y_pred - y) / len(X)  # MSE 对输出的梯度
    grad_W, grad_b = neuron.backward(grad_output)

    # 更新权重
    neuron.W = neuron.W - lr * grad_W
    neuron.b = neuron.b - lr * grad_b

    if (epoch + 1) % 20 == 0:
        print(f"  第 {epoch+1:3d} 轮：损失 = {loss.item():.4f}")

# 验证
print("\n训练完成！验证结果：")
y_final = neuron.forward(X)
for i in range(4):
    pred = 1 if y_final[i].item() > 0.5 else 0
    true = int(y[i].item())
    status = "✓" if pred == true else "✗"
    print(f"  ({int(X[i,0])}, {int(X[i,1])}) → 预测={pred}, 真实={true} {status}")

---
## 5. 两层网络的反向传播

单个神经元只有一层，反向传播很简单。现在加上隐藏层，看看梯度是怎么「穿越」隐藏层传回去的。

```
输入 x
  │
  ↓
[Layer1: z1 = x·W1 + b1, a1 = sigmoid(z1)]  ← 隐藏层
  │
  ↓
[Layer2: z2 = a1·W2 + b2, a2 = sigmoid(z2)]  ← 输出层
  │
  ↓
损失 L = MSE(a2, y)
```

反向传播的关键：**从输出层开始，逐层往回算梯度**。

1. 输出层：$\frac{\partial L}{\partial W_2}$ — 直接从损失算
2. 隐藏层：$\frac{\partial L}{\partial W_1}$ — 需要通过 $W_2$ 把梯度传回来

这就像追责：先问质检的责任，再通过质检追溯到工人。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

def sigmoid_grad(a):
    """Sigmoid 导数：a * (1-a)"""
    return a * (1 - a)

class TwoLayerMLP:
    """两层 MLP：手动实现前向传播和反向传播

    结构：输入(2) → 隐藏层(4, Sigmoid) → 输出(1, Sigmoid)
    """

    def __init__(self, input_size, hidden_size, output_size):
        """初始化权重（随机）"""
        # 第一层权重和偏置
        self.W1 = torch.randn(input_size, hidden_size)  # [2, 4]
        self.b1 = torch.randn(1, hidden_size)  # [1, 4]

        # 第二层权重和偏置
        self.W2 = torch.randn(hidden_size, output_size)  # [4, 1]
        self.b2 = torch.randn(1, output_size)  # [1, 1]

        # 中间值（反向传播时需要）
        self.x = None
        self.z1 = None
        self.a1 = None
        self.z2 = None
        self.a2 = None

    def forward(self, X):
        """前向传播：x → z1 → a1 → z2 → a2"""
        self.x = X.clone()  # 保存输入

        # 第一层
        self.z1 = torch.matmul(X, self.W1) + self.b1  # [batch, 4]
        self.a1 = sigmoid(self.z1)  # [batch, 4]

        # 第二层
        self.z2 = torch.matmul(self.a1, self.W2) + self.b2  # [batch, 1]
        self.a2 = sigmoid(self.z2)  # [batch, 1]

        return self.a2

    def backward(self, y):
        """反向传播：从输出层到隐藏层，逐层计算梯度
        参数: y — 真实标签 [batch, 1]
        """
        n = len(y)  # 样本数

        # ===== 输出层梯度 =====
        # ∂L/∂a2 = 2*(a2 - y)/n （MSE 损失对输出的梯度）
        dL_da2 = 2 * (self.a2 - y) / n

        # ∂L/∂z2 = ∂L/∂a2 × σ'(z2) = ∂L/∂a2 × a2*(1-a2)
        dL_dz2 = dL_da2 * sigmoid_grad(self.a2)  # [batch, 1]

        # ∂L/∂W2 = a1^T × ∂L/∂z2
        dL_dW2 = torch.matmul(self.a1.T, dL_dz2)  # [4, 1]

        # ∂L/∂b2 = sum(∂L/∂z2)
        dL_db2 = dL_dz2.sum(dim=0, keepdim=True)  # [1, 1]

        # ===== 隐藏层梯度（通过 W2 传回来）=====
        # ∂L/∂a1 = ∂L/∂z2 × W2^T （梯度通过 W2 传回隐藏层）
        dL_da1 = torch.matmul(dL_dz2, self.W2.T)  # [batch, 4]

        # ∂L/∂z1 = ∂L/∂a1 × σ'(z1)
        dL_dz1 = dL_da1 * sigmoid_grad(self.a1)  # [batch, 4]

        # ∂L/∂W1 = x^T × ∂L/∂z1
        dL_dW1 = torch.matmul(self.x.T, dL_dz1)  # [2, 4]

        # ∂L/∂b1 = sum(∂L/∂z1)
        dL_db1 = dL_dz1.sum(dim=0, keepdim=True)  # [1, 4]

        return dL_dW1, dL_db1, dL_dW2, dL_db2

print("TwoLayerMLP 创建完成")
print("  forward(X) — 前向传播")
print("  backward(y) — 反向传播，返回四个梯度")

### 用两层 MLP 训练 XOR

AND 是线性可分的，单个神经元就够。现在来挑战 XOR——必须用两层网络。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

def sigmoid_grad(a):
    """Sigmoid 导数"""
    return a * (1 - a)

class TwoLayerMLP:
    """两层 MLP"""
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = torch.randn(input_size, hidden_size)
        self.b1 = torch.randn(1, hidden_size)
        self.W2 = torch.randn(hidden_size, output_size)
        self.b2 = torch.randn(1, output_size)
        self.x = None
        self.z1 = self.a1 = self.z2 = self.a2 = None

    def forward(self, X):
        """前向传播"""
        self.x = X.clone()
        self.z1 = torch.matmul(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = torch.matmul(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, y):
        """反向传播"""
        n = len(y)
        # 输出层梯度
        dL_dz2 = 2 * (self.a2 - y) / n * sigmoid_grad(self.a2)
        dL_dW2 = torch.matmul(self.a1.T, dL_dz2)
        dL_db2 = dL_dz2.sum(dim=0, keepdim=True)
        # 隐藏层梯度（通过 W2 传回来）
        dL_dz1 = torch.matmul(dL_dz2, self.W2.T) * sigmoid_grad(self.a1)
        dL_dW1 = torch.matmul(self.x.T, dL_dz1)
        dL_db1 = dL_dz1.sum(dim=0, keepdim=True)
        return dL_dW1, dL_db1, dL_dW2, dL_db2

# XOR 数据
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 创建模型
mlp = TwoLayerMLP(input_size=2, hidden_size=4, output_size=1)
lr = 3.0  # 学习率

print("训练 XOR（两层 MLP + 手动反向传播）")
print("=" * 55)

losses = []
for epoch in range(500):
    # 前向传播
    y_pred = mlp.forward(X)

    # 计算损失
    loss = torch.mean((y_pred - y) ** 2)
    losses.append(loss.item())

    # 反向传播
    dL_dW1, dL_db1, dL_dW2, dL_db2 = mlp.backward(y)

    # 更新权重
    mlp.W1 = mlp.W1 - lr * dL_dW1
    mlp.b1 = mlp.b1 - lr * dL_db1
    mlp.W2 = mlp.W2 - lr * dL_dW2
    mlp.b2 = mlp.b2 - lr * dL_db2

    if (epoch + 1) % 100 == 0:
        print(f"  第 {epoch+1} 轮：损失 = {loss.item():.4f}")

# 验证
print("\n验证结果：")
y_final = mlp.forward(X)
for i in range(4):
    pred = 1 if y_final[i].item() > 0.5 else 0
    true = int(y[i].item())
    status = "✓" if pred == true else "✗"
    print(f"  ({int(X[i,0])}, {int(X[i,1])}) → 预测={pred}, 真实={true} {status}")

---
## 6. 翻译词典

| 生活直觉 | 深度学习术语 | 英文 |
|----------|-------------|------|
| 客户投诉后逐层追责 | 信用分配 | Credit Assignment |
| 多米诺骨牌连锁反应 | 链式法则 | Chain Rule |
| 从后往前逐层追责 | 反向传播 | Backpropagation |
| 从前往后算结果 | 前向传播 | Forward Pass |
| 每个环节应承担的责任 | 梯度 | Gradient |
| 通过质检追溯到工人 | 梯度穿越隐藏层 | Gradient Flow Through Hidden Layer |

---
## 7. 总结与预告

### 今日核心收获

1. **信用分配问题**：输出误差需要分摊到每个权重
2. **链式法则**：$\frac{\partial L}{\partial A} = \frac{\partial L}{\partial B} \times \frac{\partial B}{\partial A}$
3. **反向传播**：从输出层到输入层，逐层计算梯度
4. **隐藏层的梯度**：通过 $W_2^T$ 把梯度从输出层传回隐藏层

### 关键流程

```
前向传播：x → z1 → a1 → z2 → a2 → Loss
反向传播：Loss → ∂L/∂a2 → ∂L/∂z2 → ∂L/∂W2
                       ↘ ∂L/∂a1 → ∂L/∂z1 → ∂L/∂W1
```

### 明天预告

今天我们手写了反向传播的每一步。这很累，但很有价值——你理解了底层原理。

明天用 PyTorch 的 **autograd** 自动完成这一切，实现一个完整的 MLP 训练流程，彻底解决 XOR 问题！